## Kontrol Yapıları ve Fonksiyonlar

- Motivasyon: Buradaki 'bazı' konular (örn. karar yapıları/döngüler/fonksiyon tanımlama) aslında başka programlama dillerini çalışırken defalarca gördüğüm ve aşina olduğum şeyler. Yine de bildiklerimi cilalamak ve Python syntaxına daha da aşinalık kazanmak için pratiklerini yapmamım iyi olacağını düşündüm.
- Soruları nereden bulduğumu, diğer detayları 'p2_veri_tipleri' dosyasında anlatmıştım. Soruların zorluklarını 5 üzerinden puanlıyorum aynı şekilde
- Closures, Decorators gibi daha zor konuları diğer 'p3' dosyasında çalıştım. 

### 1 - Koşullu İfadeler (1/5)

Bağlam: Bir web uygulamasının yetkilendirme (authorization) modülünü yazıyoruz. Sisteme giriş yapmaya çalışan kullanıcıların hesap durumlarına ve atanan rollerine göre farklı sayfalara yönlendirilmesi veya engellenmesi gerekiyor. Gereksinimler:
1. erisim_kontrolu(rol: str, hesap_aktif_mi: bool) adında bir fonksiyon tanımla.
2. Önceki konularda öğrendiğimiz "truthiness" kuralını kullanarak, eğer rol boş bir string ( "" ) ise if not ile yakalayıp "Rol belirtilmedi" hatası döndür (erken çıkış / early return).
3. Hesabın aktif olup olmadığını kontrol et; hesap pasifse ( False ), rolün ne olduğuna bakılmaksızın doğrudan "Hesap askıda" uyarısı döndür.
4. Eğer hesap aktifse, if/elif/else zinciri kurarak sırasıyla "admin" için tam yetki, "editor" için içerik yetkisi, "standart" için okuma yetkisi mesajları döndür. Gelen rol metnini önceki konulardaki gibi .strip().lower() ile temizle.

In [14]:
# Test Verisi
testler = [
    (" ADMIN ", True),      # Başarılı - Admin (Boşluklu ve büyük harf)     
    ("editor", True),       # Başarılı - Editör
    ("standart", True),     # Başarılı - Standart
    ("misafir", True),      # Başarısız - Bilinmeyen rol
    ("admin", False),       # Başarısız - Rol yetkili ama hesap pasif     
    ("", True)              # Başarısız - Rol boş (Edge-case)
]


def erisim_kontrolu(rol: str, hesap_aktifligi: bool) -> str:
    
    rol_clean = rol.strip().lower()

    # Edge-case 1
    if not rol:
        return "Başarısız - Rol belirtilmedi"
        
    # Edge-case 2
    if not hesap_aktifligi:
        return "Başarısız - Hesap askıda"
        

    if rol_clean == "admin":        return "Başarılı - Tam yetki"
    elif rol_clean == "editor":     return "Başarılı - İçerik yetkisi"
    elif rol_clean == "standart":   return "Başarılı - Okuma yetkisi"
    else:                           return"Başarısız - Bilinmeyen rol"


for rol, hesap_aktifligi in testler:
    sonuc = erisim_kontrolu(rol=rol, hesap_aktifligi=hesap_aktifligi)
    print(f"Rol: {repr(rol)}, Aktif: {hesap_aktifligi} -> {sonuc}")


Rol: ' ADMIN ', Aktif: True -> Başarılı - Tam yetki
Rol: 'editor', Aktif: True -> Başarılı - İçerik yetkisi
Rol: 'standart', Aktif: True -> Başarılı - Okuma yetkisi
Rol: 'misafir', Aktif: True -> Başarısız - Bilinmeyen rol
Rol: 'admin', Aktif: False -> Başarısız - Hesap askıda
Rol: '', Aktif: True -> Başarısız - Rol belirtilmedi


### 2 - Döngüler (3/5)

Bağlam: Bir IoT (Nesnelerin İnterneti) sisteminde, sensörlerden gelen sıcaklık verilerini işleyen ve cihaz bağlantı kopukluklarını yöneten bir kontrol modülü yazıyoruz.
Gereksinimler:
1. İçinde sayısal değerler ve iletişim hatalarını simüle eden None değerleri barındıran bir listeyi for döngüsü ile tara.
2. Döngü içinde, değer None ise veya 0'dan küçük mantıksız bir değerse (sensör hatası) işlemi continue ile atla.
3. Sıcaklık 100.0 derecenin üzerindeyse, yangın riski taşıdığı için döngüyü break ile tamamen durdur ve acil durum bayrağı (flag) döndür.
4. Ayrı bir fonksiyonda, sensör bağlantısı koptuğunda yeniden bağlanmayı denemek için bir while döngüsü kur; maksimum deneme sayısına (örn: 3) ulaşıldığında döngüyü kırıp başarısızlık mesajı ver.
5. Edge-case: Veri listesi boş gönderilirse erken çıkış yaparak uyarı döndür.

In [ ]:
# Test Verisi
test_okumalari = [22.5, None, -5.0, 45.0, 105.5, 30.2]
test_listeleri = [[],[],test_okumalari,[]]


# Gereksinim 1-3 ve 5: --- Sensör Analiz Yapan Fonksiyon ---
def sensor_analizi_yap(veriler: list[float | None]) -> bool:

    if not veriler:         # Edge-case: Boş veri seti kontrolü
        print("Uyarı: İşlenecek sensör verisi bulunamadı.")
        return False

    acil_durum = False
    print("--- Sıcaklık Analizi Başlıyor ---")
    for deger in veriler:

        # Hatalı ölçümleri atla
        if deger is None or deger < 0:
            print(f"Bozuk/eksik veri atlandı: {deger}")
            continue

        # Kritik eşik kontrolü
        if deger > 100.0:
            print(f"Kritik Uyarı! Sıcaklık çok yüksek ({deger}C). Acil duruma geçiliyor.")
            acil_durum = True
            break

        print(f"Normal ölçüm: {deger}C")

    return acil_durum


# Gereksinim 4: --- Yeniden bağlantı deneme Fonksiyonu --- 
def baglanti_dene(veri_seti: list[list[float|None]], max_deneme: int = 3) -> list[float|None] | None:

    deneme = 1
    while deneme <= max_deneme and deneme<=len(veri_seti):
        if not veri_seti[deneme-1]:  
            print(f"Veri bulunamadı! Bağlantı yeniden kuruluyor... \n(Deneme {deneme}/{max_deneme})")
            deneme += 1
        else:
            print(f"Veri bulundu! \n(Deneme {deneme}/{max_deneme})")
            return veri_seti[deneme-1]
        
    if deneme > len(veri_seti):
        print("Kontrol edilecek veri paketi kalmadı.")
    else:
        print("Maksimum deneme sayısına ulaşıldı.")   
    return None


# --- Yazılan Fonksiyonların Sınanması ---
saglam_veriler = baglanti_dene(test_listeleri, max_deneme=3)

if saglam_veriler:
    if sensor_analizi_yap(saglam_veriler):
        print("Acil durum modu aktifleştirilmiş. Sıra dışı değer var.")
    else:
        print("Tüm değerler normal")
else:
    print("Bütün bağlantı denemeleri başarısız olmuş.")


Veri bulunamadı! Bağlantı yeniden kuruluyor... 
(Deneme 1/3)
Veri bulunamadı! Bağlantı yeniden kuruluyor... 
(Deneme 2/3)
Veri bulundu! 
(Deneme 3/3)
--- Sıcaklık Analizi Başlıyor ---
Normal ölçüm: 22.5C
Bozuk/eksik veri atlandı: None
Bozuk/eksik veri atlandı: -5.0
Normal ölçüm: 45.0C
Kritik Uyarı! Sıcaklık çok yüksek (105.5C). Acil duruma geçiliyor.
Acil durum modu aktifleştirilmiş. Sıra dışı değer var.


### 3 - Iterators ve Generators (3/5)

Bağlam: Bir sunucu çiftliğinde günde gigabaytlarca üretilen log (günlük) dosyalarını analiz etmemiz gerekiyor. Dosyanın tamamını bir listeye alıp belleğe (RAM) yüklemek sistemi çökerteceği için, veriyi yalnızca ihtiyaç anında satır satır üreten bellek dostu bir yapı (generator pipeline) kuracağız.
Gereksinimler:
1. Girdi olarak aldığı veriyi (gerçekte dev bir dosya olduğunu varsayalım) satır satır okuyan ve return yerine yield anahtar kelimesini kullanarak bir üreteç (generator) oluşturan log_okuyucu fonksiyonu yaz.
2. İlk üreteçten gelen akışı (iterator) girdi olarak alan ve yalnızca içinde "ERROR" kelimesi geçen satırları yield eden ikinci bir hata_filtresi üreteci yazarak bir "veri boru hattı (pipeline)" kur.
3. Ana programda, arka planda bu sistemin nasıl çalıştığını göstermek için küçük bir listeyi iter() fonksiyonuyla yineleyiciye
(iterator) çevir ve elemanlarını next() fonksiyonuyla manuel olarak tek tek çek.
4. Kurduğun boru hattını kullanarak loglardan sadece ilk 2 hatayı çekip break ile işlemi tamamen durdur.

In [ ]:
# Test Verisi 
dev_log_verisi = [
        "INFO: Sistem başlatıldı.",
        "WARNING: Bellek kullanımı %80.",
        "ERROR: Veritabanı bağlantısı koptu.",
        "INFO: Yeniden bağlanılıyor...",
        "ERROR: Kimlik doğrulama başarısız (Timeout).",        
        "DEBUG: Kullanıcı oturumu kapatıldı.",
        "ERROR: Disk alanı yetersiz."
    ]

# Generator 1 - Parametre olarak liste alır. Generator döndürür.
def log_okuyucu(log_satirlari):
    index = 0
    while index < len(log_satirlari):
        yield log_satirlari[index]
        index += 1

# Generator 2 - Parametre olarak bir iterator/generator nesnesi alır. Generator döndürür.
def hata_filtresi(okuyucu):
    for log_satiri in okuyucu:
        if "ERROR" in log_satiri:
            yield log_satiri.strip()


# --- Yazdığımız generatorlerin sınanması / Pipeline hata taraması ---
log_okuyucumuz = log_okuyucu(dev_log_verisi)
filtrelenmis_okuyucu = hata_filtresi(log_okuyucumuz)

bulunan_hata = 0
for hata_satiri in filtrelenmis_okuyucu:
    print("Tespit:", hata_satiri)
    bulunan_hata += 1
    if bulunan_hata == 2:
        print("İlk 2 hata bulundu, durduruluyor!")
        break

Tespit: ERROR: Veritabanı bağlantısı koptu.
Tespit: ERROR: Kimlik doğrulama başarısız (Timeout).
İlk 2 hata bulundu, durduruluyor!


### 4 - Iterators ve Generators (3/5)

Bağlam: Bir bankacılık sisteminde, gerçek zamanlı işlem (transaction) akışını analiz ediyorsun. Günlük milyonlarca işlem olduğu için tüm veriyi belleğe yüklemek yerine, işlemleri tek tek, ihtiyaç anında üreten (lazy) bir yapı kurman gerekiyor. Ayrıca belirli bir tutarın üzerindeki işlemleri "şüpheli" olarak işaretleyip ilk birkaçını hızlıca yakalayan bir uyarı sistemi yazacaksın.

Gereksinimler:

1. İşlem tutarlarını içeren bir listeyi yield kullanarak tek tek üreten bir üreteç (generator) fonksiyonu yaz.
2. İlk üreteçten gelen akışı (iterator) girdi olarak alan, yalnızca 10.000 TL'nin üzerindeki tutarları yield eden ikinci bir üreteç yazarak bir "boru hattı" (pipeline) kur.. Bu filtre içinde None veya negatif değerleri de güvenle atlamalısın.
3. Kurduğun boru hattını kullanarak işlem akışından yalnızca ilk 2 şüpheli işlemi yakala, ikinciyi bulduğunda break ile taramayı tamamen durdur (böylece geri kalan veriler hiç işlenmemiş/belleğe alınmamış olsun).
4. Ana programda, küçük bir örnek liste üzerinde iter() ile bir yineleyici oluştur ve next() ile elemanlarını manuel olarak tek tek çek. Yineleyici tükendiğinde fırlatılacak StopIteration hatasını bir try/except bloğuyla yakala.

In [36]:
# Test Verisi
gunluk_islemler = [1500.0, None, -200.0, 12500.0, 8000.0, None, 25000.0, 3000.0, 45000.0] 


def islem_uretici(islemler):
    for islem in islemler:
        yield islem

def supheli_filtre(ham_islemler):
    for islem in ham_islemler:
        if islem is None or islem < 0:
            continue
        if islem > 10_000: 
            yield islem

# --- Tanımladığımız generatorlerin test edilmesi / Pipeline ---
ham_islem_okuyucu = islem_uretici(gunluk_islemler)
supheli_islem_okuyucu = supheli_filtre(ham_islem_okuyucu)

okunan_islem = 0
for supheli_islem in supheli_islem_okuyucu:
    print("Şüpheli islem:", supheli_islem)
    okunan_islem += 1
    if okunan_islem == 2:
        print("Yeterli sayıda şüpheli işlem yakalandı")
        break


# --- Gereksinim 3: Stop Iteration Hatası --- 
gunluk_islemler_iter = iter(gunluk_islemler)
print("\n --- Tüm İşlemler ---")
while True:
    try:
        print(next(gunluk_islemler_iter), end=", ")
    except StopIteration as hata_mesaji:
        print("\nHata mesajı: Yineleyici tükendi, elde edilecek başka eleman kalmadı.")
        break

Şüpheli islem: 12500.0
Şüpheli islem: 25000.0
Yeterli sayıda şüpheli işlem yakalandı

 --- Tüm İşlemler ---
1500.0, None, -200.0, 12500.0, 8000.0, None, 25000.0, 3000.0, 45000.0, 
Hata mesajı: Yineleyici tükendi, elde edilecek başka eleman kalmadı.


### 5 - match-case Yapısı (2/5)

Bağlam: Bir mikroservis mimarisinde, dış sistemlerden gelen farklı yapıdaki JSON veri paketlerini (payload) içerdikleri alanlara (keys) ve değerlere göre yönlendiren bir API yönlendiricisi (router) yazıyoruz.
Gereksinimler:
1. Gelen bir sözlüğü (dict) parametre olarak alan ve match ifadesi kullanarak analiz eden bir fonksiyon yaz.
2. İlk case durumunda, gelen veri {"aksiyon": "giris", "kullanici": k} yapısındaysa, kullanıcı adı "admin" olduğunda devreye girecek bir eşleştirme (literal matching) yapıp özel bir yönetici mesajı döndür.
3. İkinci durumda, aynı yapıyı kullanıp k değişkenini yakalayarak (capture) normal kullanıcılar için dinamik bir karşılama mesajı döndür.
4. Üçüncü durumda, {"aksiyon": "veri_cek", "endpoint": e, "limit": l} desenini eşleştirip bir veri çekme onayı döndür.
5. Hiçbir desene uymayan veya eksik alan içeren veriler (edge-case) için case _: (wildcard) kullanarak geçersiz paket uyarısı ver.

In [ ]:
#Test Verisi
istekler = [
    {"aksiyon": "giris", "kullanici": "admin"},                   
    {"aksiyon": "giris", "kullanici": "veri_uzmani_99"},               
    {"aksiyon": "veri_cek", "endpoint": "/sensorler", "limit": 100},     
    {"aksiyon": "sil", "hedef": "kullanici_12"},                  
    {"aksiyon": "giris"}                                         
]

def veri_yonlendirici(veri: dict) -> None:
    match veri:
        case {"aksiyon":"giris", "kullanici": "admin"}:     # Sabit Değer Eşleme
            print("'admin' kullanıcısı 'giris' eylemini gerçekleştirdi.")
        case {"aksiyon":"giris", "kullanici": k}:           # Değişken Eşleme
            print(f"'{k}' kullanıcısı 'giris' eylemini gerçekleştirdi.")
        case {"aksiyon":"veri_cek", "endpoint": e, "limit": l}:        
            print(f"Veri tabebi alındı. '{e}' konumundan '{l}' adet kayıt çekilecek.")
        case _:
            print("Geçersiz paket!")


for veri in istekler:
    veri_yonlendirici(veri)    

'admin' kullanıcısı 'giris' eylemini gerçekleştirdi.
'veri_uzmani_99' kullanıcısı 'giris' eylemini gerçekleştirdi.
Veri tabebi alındı. '/sensorler' konumundan '100' adet kayıt çekilecek.
Geçersiz paket!
Geçersiz paket!


### 6 - Fonksiyon Tanımı, Parametre Çeşitleri (2/5)

Bağlam: Bir kütüphane otomasyon sisteminde, kitap ödünç verme işlemi için gecikme cezası hesaplayan bir fonksiyon yazıyorsun.

Gereksinimler:

1. gecikme_cezasi_hesapla(gecikme_gunu, gunluk_ucret=2.50, ust_sinir=50.0) fonksiyonu tanımla; ceza gecikme_gunu * gunluk_ucret olarak hesaplanır, ancak ust_sinir'i aşamaz.
2. gecikme_gunu negatifse ValueError fırlat (anlamlı bir mesajla).
3. gecikme_gunu 0 ise ceza 0.0 dönmeli (varsayılan değerlerle bile fonksiyon çağrılabilir olmalı).
4. Fonksiyonu hem yalnızca zorunlu parametreyle, hem pozisyonel, hem de anahtar kelime (keyword) argümanlarıyla çağırarak farklı senaryoları test et.
5. ogrenci_indirimi adında varsayılanı False olan bir bool parametre ekle; True ise hesaplanan (ve üst sınırla sınırlanan) cezanın üzerine ek olarak %20 indirim uygula.
6. Ceza her zaman 2 ondalık basamağa yuvarlanarak döndürülmeli.

In [14]:
# Test Verisi
# Senaryo 1: yalnızca zorunlu parametre
gecikme_gunu_1 = 5

# Senaryo 2: varsayılanları değiştirerek, öğrenci indirimiyle
gecikme_gunu_2 = 10
gunluk_ucret_2 = 3.0
ust_sinir_2 = 20.0

# Senaryo 3: üst sınırı aşan durum
gecikme_gunu_3 = 100

# Senaryo 4: sıfır gecikme
gecikme_gunu_4 = 0

# Senaryo 5: geçersiz girdi
gecikme_gunu_5 = -3

# Yazdığım Fonksiyon
def gecikme_cezasi_hesapla(gecikme_gunu, gunluk_ucret=2.50, ust_sinir=50.0, ogrenci_indirimi= False):

    if gecikme_gunu < 0:
        raise ValueError(f"Gecikme günü değerini lütfen pozitif giriniz! (Sizin girmiş olduğunuz değer: {gecikme_gunu})")

    gecikme_cezasi = gecikme_gunu * gunluk_ucret
    gecikme_cezasi = ust_sinir if gecikme_cezasi >= ust_sinir else gecikme_cezasi
    # Şu daha mantıklı -> gecikme_cezasi = min(gecikme_cezasi, ust_sinir)
    if ogrenci_indirimi: gecikme_cezasi *= 0.8

    return gecikme_cezasi


cezalar = list()
cezalar.append(gecikme_cezasi_hesapla(gecikme_gunu_1))
cezalar.append(gecikme_cezasi_hesapla(gecikme_gunu_2, gunluk_ucret_2, ust_sinir_2, ogrenci_indirimi=True))
cezalar.append(gecikme_cezasi_hesapla(gecikme_gunu_3))
cezalar.append(gecikme_cezasi_hesapla(gecikme_gunu_4))

for i, sonuc in enumerate(cezalar, 1):
    print(f"Senaryo {i}: Hesaplanan Ceza: {sonuc}")

try:
    print("Senaryo 5:", end=" ")
    gecikme_cezasi_hesapla(gecikme_gunu_5)
except ValueError as hata_mesaji:
    print(f"Hata Mesajı: {hata_mesaji}")


Senaryo 1: Hesaplanan Ceza: 12.5
Senaryo 2: Hesaplanan Ceza: 16.0
Senaryo 3: Hesaplanan Ceza: 50.0
Senaryo 4: Hesaplanan Ceza: 0.0
Senaryo 5: Hata Mesajı: Gecikme günü değerini lütfen pozitif giriniz! (Sizin girmiş olduğunuz değer: -3)


### 7 - *args, **kwargs ve Esnek Parametre Yapıları (5/5)

Bağlam: Bir log toplama aracında, değişken sayıda etiket ve değişken sayıda meta veri alanıyla log kaydı oluşturan esnek bir fonksiyon yazıyorsun. Gereksinimler:

1. log_kaydi_olustur(seviye, \*etiketler, kaynak="sistem", \*\*meta_veri) fonksiyonu tanımla; seviye zorunlu pozisyonel parametre, etiketler değişken sayıda pozisyonel argüman (\*args), kaynak keyword-only benzeri kullanılan varsayılanlı parametre, meta_veri ise değişken sayıda anahtar-değer çifti (\*\*kwargs) olsun.
2. Fonksiyon, tüm bilgileri tek bir formatlı string olarak döndürsün: "[SEVIYE] (kaynak) etiket1,etiket2 | anahtar1=deger1, anahtar2=deger2" biçiminde; etiketler boşsa "etiketsiz" yazsın, meta_veri boşsa meta kısmı hiç eklenmesin.
3. seviye şu değerlerden biri olmalı: "DEBUG", "INFO", "WARNING", "ERROR"; değilse ValueError fırlatsın.
4. Ayrı bir toplu_log_isle(*log_kayitlari, **ortak_meta) fonksiyonu yaz: pozisyonel argüman olarak gelen her bir log string'ini işlerken, ortak_meta içindeki anahtar-değer çiftlerini her birine ek olarak (sondan) iliştirsin.
5. meta_veri içinde seviye veya kaynak adında bir anahtar geçmeye çalışılırsa (çakışma riski), bunu tespit edip anlamlı bir TypeError fırlatsın (fonksiyonun kendiliğinden fırlattığı çakışma hatasına güvenme, açıkça kontrol et).
6. Argümanları hem yalnızca zorunlu parametreyle hem de tüm esnek yapıları doldurarak çağıran örnekler göster.

In [ ]:
# --- TEST VERİSİ --- 
# Senaryo 1: minimum çağrı
seviye_1 = "INFO"

# Senaryo 2: tam dolu çağrı
seviye_2 = "ERROR"
etiketler_2 = ("veritabani", "baglanti")
kaynak_2 = "auth-service"
meta_2 = {"kullanici_id": 42, "deneme_sayisi": 3}

# Senaryo 3: geçersiz seviye
seviye_3 = "TRACE"

# Senaryo 4: meta_veri içinde çakışan anahtar
meta_4 = {"seviye": "DEBUG"}

# Senaryo 5: toplu işleme
kayitlar = ["[INFO] (sistem) etiketsiz", "[ERROR] (db) baglanti"]
ortak_meta = {"oturum_id": "abc-123"}



# --- ÇÖZÜM ----

GECERLI_SEVIYELER = {"DEBUG", "INFO", "WARNING", "ERROR"}


def log_kaydi_olustur(seviye, *etiketler, kaynak="sistem", **metaveri) -> str:

    # Edge-case 
    if seviye not in GECERLI_SEVIYELER:
        raise ValueError(f"Geçersiz seviye: '{seviye}'. Beklenen: {GECERLI_SEVIYELER}")

    # Edge-case 2
    cakisan_anahtarlar = metaveri.keys() & {"seviye", "kaynak"}     # Python'un kendi uyarısı daha önce çalışıyor zaten :(
    if cakisan_anahtarlar:
        raise TypeError(f"'meta_veri' ayrılmış anahtar(lar) içeremez: {cakisan_anahtarlar}")

    etiket_str = ",".join(etiketler) if etiketler else "etiketsiz"
    satir = f"[{seviye}] ({kaynak}) {etiket_str}"

    if metaveri:
        meta_str = ", ".join(f"{key}={val}" for key, val in metaveri.items())  # Generator Expression ile 
        satir += f" | {meta_str}"

    return satir 


def toplu_log_isle(*log_kayitlari, **ortak_meta):

    if not ortak_meta:
        return list(log_kayitlari)

    ek_meta_str = ", ".join(f"{key}={val}" for key, val in ortak_meta.items())
    return [f"{kayit} | {ek_meta_str}" for kayit in log_kayitlari]  # List Comprehension


log_kayitlari = []
log_kayitlari.append(log_kaydi_olustur(seviye_1))
log_kayitlari.append(log_kaydi_olustur(seviye_2, *etiketler_2, kaynak=kaynak_2, **meta_2))

try:
    log_kaydi_olustur(seviye_3)
except ValueError as hata_mesaji:
    print(f"Hata Mesajı: {hata_mesaji}")

try:
    log_kaydi_olustur(seviye_2, *etiketler_2, kaynak=kaynak_2, **meta_4)
except TypeError as hata_mesaji:
    print(f"Hata Mesajı: {hata_mesaji}")


log_kayitlari.extend(kayitlar)
log_kayitlari_toplu = toplu_log_isle(*log_kayitlari, **ortak_meta)
for log_kaydi in log_kayitlari_toplu:
    print(log_kaydi)


Hata Mesajı: Geçersiz seviye: 'TRACE'. Beklenen: {'DEBUG', 'INFO', 'ERROR', 'WARNING'}
Hata Mesajı: log_kaydi_olustur() got multiple values for argument 'seviye'
[INFO] (sistem) etiketsiz | oturum_id=abc-123
[ERROR] (auth-service) veritabani,baglanti | kullanici_id=42, deneme_sayisi=3 | oturum_id=abc-123
[INFO] (sistem) etiketsiz | oturum_id=abc-123
[ERROR] (db) baglanti | oturum_id=abc-123


### 8 - Scope Kuralları ve Closurelara Giriş (5/5)

* Anlaması çok zor bi konu bu closure + decorators. Cidden :( 
* OOP state encapsulation vs. görmemiş olsam hiç anlayamayacaktım ki. Neyse yapay zekadaki çözüm güzel açıklamış.

Bağlam: Bir oyun motorunda, her düşman karakter için bağımsız bir sağlık takip fonksiyonu üreten bir "fabrika" (factory) yapısı kuruyorsun; her closure kendi kapalı (enclosed) durumunu korumalı. Gereksinimler:

1. saglik_takipcisi_olustur(baslangic_can) adında bir fonksiyon yaz; bu fonksiyon içeride bir mevcut_can değişkeni tutan ve hasar_al(miktar) adında iç içe (nested) bir fonksiyon döndüren bir closure olsun.
2. hasar_al(miktar) çağrıldığında mevcut_can'ı azaltmalı (nonlocal kullanarak) ve güncel can miktarını döndürmeli; can 0'ın altına düşemez, 0'da sabitlenmeli.
3. Aynı fabrikadan üretilen iki farklı closure'ın (dusman1, dusman2) birbirinden tamamen bağımsız durumu koruduğunu göster (birine hasar verilmesi diğerini etkilememeli).
4. miktar negatifse (hasar_al negatif hasar = iyileşme anlamına gelmemeli) ValueError fırlat; global bir sayaç (toplam_gecersiz_deneme) global anahtar kelimesiyle artırılsın.
5. Closure'ın güncel mevcut_can değerini döndüren, parametresiz ayrı bir can_durumu() iç fonksiyonu da ekle; saglik_takipcisi_olustur artık (hasar_al, can_durumu) tuple'ı döndürsün.
6. Bir for döngüsü içinde birden fazla closure oluşturup geç bağlama (late binding) tuzağına düşmeden her birinin kendi baslangic_can değerini doğru şekilde kapsadığını kanıtla (klasik döngü-closure hatasını gösterip düzeltilmiş haliyle çöz).

In [ ]:
# Test Verisi
baslangic_can_1 = 100
baslangic_can_2 = 50
hasarlar_1 = [20, 15, 200]  # son hasar 0'ın altına düşürmeye çalışacak
hasarlar_2 = [-10]  # geçersiz (negatif) hasar denemesi
can_degerleri_liste = [80, 120, 60]  # late-binding testi için 


# Çözüm
toplam_gecersiz_deneme = 0

def saglik_takipcisi_olusturucu(baslangic_can):
    mevcut_can = baslangic_can

    def hasar_al(hasar):
        # Best Practice olarak global/nonlocal bildirimleri en başta yapılmalıymış.
        global toplam_gecersiz_deneme
        nonlocal mevcut_can

        if hasar < 0:
            toplam_gecersiz_deneme += 1 
            raise ValueError("Alınan hasar negatif olamaz")
        
        mevcut_can -= hasar
        mevcut_can = max(0, mevcut_can)
        return mevcut_can
    
    def can_durumu():
        return mevcut_can   # Okuma yaptığımız için nonlocal'e gerek yok
    
    return hasar_al, can_durumu

# Aynı fonksiyondan üretilen ayrı çocuk fonksiyonların ayrı stateler tutabilmesi
hasar_al_1, can_durumu_1 = saglik_takipcisi_olusturucu(baslangic_can_1)
hasar_al_2, can_durumu_2 = saglik_takipcisi_olusturucu(baslangic_can_2)

print(f"Toplam geçersiz deneme: {toplam_gecersiz_deneme}")

for hasar in hasarlar_1:
    print(f"dusman1 can: (Alınan hasar:{hasar}) {hasar_al_1(hasar)}")

print(f"dusman2 can (negatif hasardan etkilenmemeli): {can_durumu_2()}")
try:
    hasar_al_2(hasarlar_2[0])
except ValueError as hata_mesaji:
    print(f"Hata: {hata_mesaji}")
     
print(f"Toplam geçersiz deneme: {toplam_gecersiz_deneme}")


# Late Binding
hatali_closurelar = []
for can in can_degerleri_liste:
    def can_goster():           # Burada direkt can değişkeninin referansını kullanıyor. (Hepsi için aynı nesneyi son atanan değeriyle gösterilecek döngü sonunda)
        return can
    hatali_closurelar.append(can_goster)
print("Hatalı tanımlanan fonksiyonlar: ", [can_goster() for can_goster in hatali_closurelar])


# Early Binding
dogru_closurelar = []
for can in can_degerleri_liste:
    def can_goster(can=can):    # can=can diyerek can'ın value'sını can değişkenine aktarıyoruz
        return can
    dogru_closurelar.append(can_goster)
print("Doğru tanımlanan fonksiyonlar: ", [can_goster() for can_goster in dogru_closurelar])

Toplam geçersiz deneme: 0
dusman1 can: (Alınan hasar:20) 80
dusman1 can: (Alınan hasar:15) 65
dusman1 can: (Alınan hasar:200) 0
dusman2 can (negatif hasardan etkilenmemeli): 50
Hata: Alınan hasar negatif olamaz
Toplam geçersiz deneme: 1
Hatalı tanımlanan fonksiyonlar:  [60, 60, 60]
Doğru tanımlanan fonksiyonlar:  [80, 120, 60]


### 9 - Lambda Fonksiyonları ve map, filter, reduce Üçlüsü (4/5) 

Bağlam: Bir online market uygulamasında, sipariş listesi üzerinde fiyat dönüştürme, kategori filtreleme ve toplam tutar hesaplama işlemlerini map/filter/reduce ile fonksiyonel bir hatta indirgiyorsun. Gereksinimler:

1. siparisler listesindeki her sipariş dict'i için, lambda + map kullanarak fiyat alanına %18 KDV ekleyen yeni bir liste (kdv_dahil_siparisler) üret; orijinal siparisler değişmemeli.
2. lambda + filter kullanarak yalnızca kategori alanı "elektronik" olan siparişleri ayıkla; sonucu bir listeye çevir.
3. functools.reduce + lambda kullanarak elektronik siparişlerin KDV dahil toplam tutarını tek bir float olarak hesapla; boş bir listeyle çağrılırsa reduce'un fırlatacağı TypeError'ı yakalayıp 0.0 başlangıç değeriyle güvenli sonuç döndür (yani initial parametresini doğru kullan, try/except'e gerek kalmasın — ama neden gerekmediğini gösterecek şekilde yaz).
4. sorted() + lambda (key fonksiyonu olarak) kullanarak siparişleri KDV dahil fiyata göre azalan sırada sıralayan bir liste üret; eşit fiyatlarda orijinal sıra korunmalı (kararlılık/stability'i belirt).
5. Herhangi bir sipariş dict'inde fiyat anahtarı eksikse veya fiyat sayısal değilse, o sipariş map aşamasında atlanmadan hata fırlatmalı (lambda içinde tip kontrolü yaparak açık bir ValueError yükselt) — sessizce yutulmamalı.
6. En az bir sipariş kayıtta bozuk veri (fiyat eksik/hatalı) içeren ayrı bir test listesiyle bu hata yolunu göster.

In [ ]:
# Test Verisi
siparisler = [
    {"urun": "Kulaklık", "kategori": "elektronik", "fiyat": 250.0},
    {"urun": "Kitap", "kategori": "kirtasiye", "fiyat": 45.0},
    {"urun": "Telefon Kilifi", "kategori": "elektronik", "fiyat": 80.0},
    {"urun": "Defter", "kategori": "kirtasiye", "fiyat": 15.0},
    {"urun": "Kulaklık", "kategori": "elektronik", "fiyat": 250.0},  # eşit fiyat, stability testi
]

siparisler_bozuk = [
    {"urun": "Kulaklık", "kategori": "elektronik", "fiyat": 250.0},
    {"urun": "Mouse", "kategori": "elektronik", "fiyat": "yirmi"},  # hatalı tip
]


# Çözüm 
# lambda ile beraber kullanacağımız yardımcı fonksiyonumuz
def kdv_hesapla(siparis: dict) -> dict:
    fiyat = siparis.get("fiyat")
    if not isinstance(fiyat, (int, float)) or isinstance(fiyat, bool):
        raise ValueError(f"Geçersiz fiyat saptandı: {fiyat}")
    return {**siparis, "fiyat": round(fiyat*(1+0.18), 2)}     
    # Yeni sözlük döndürüyoruz. Bu satır en önemli kısım

def sozluk_listesi_yazdirma(liste: list[dict]) -> None:
    for sozluk in liste:
        print("")
        for key, val in sozluk.items():
            print(f" | {key:10}: {val:15}", end="")
    print("\n")

# 1) map() ile kdvlerini hesaplama ve 5. gereksinim olan Tip Kontrolünün map aşamasında yapılması
siparisler_kdvli = list(map(lambda sozluk: kdv_hesapla(sozluk), siparisler))
# siparisler içindeki elemanlar kaybolmasın diye list() ile kalıcı yapıyoruz 
# -> Bunu her seferinde unutuyorum

# 2) filter() ile kategorisi elektronik olanlardan ayrı bir liste oluşturma
siparisler_elektronik = list(filter(lambda sozluk: sozluk["kategori"]=="elektronik", siparisler_kdvli))

# 3) reduce() ile fiyatların toplanması ve tek bir değer döndürülmesi. Initial value of acc = 0.0 olacak
from functools import reduce
siparisler_toplam = reduce(lambda acc, siparis: acc + siparis["fiyat"], siparisler_elektronik, 0.0)

# 4) sorted() ile Azalan sırada(reverse=True) sıralanması
siparisler_sirali = sorted(siparisler_kdvli, key= lambda sozluk: sozluk["fiyat"], reverse=True)


print("Orijinal Sözlük:"); sozluk_listesi_yazdirma(siparisler)
print("KDV'li fiyatlar (map):"); sozluk_listesi_yazdirma(siparisler_kdvli)
print("Sadece Elektronik (filter):"); sozluk_listesi_yazdirma(siparisler_elektronik)
print("Fiyatları en yüksekten en düşüğe (sorted):"); sozluk_listesi_yazdirma(siparisler_sirali)
print(f"Toplam fiyat (reduce): {siparisler_toplam:.2f} TL")


try:
    bozuk_kdvli = list(map(lambda sozluk: kdv_hesapla(sozluk), siparisler_bozuk))
except ValueError as hata_mesaji:
    print(f"Hata: {hata_mesaji}")

Orijinal Sözlük:

 | urun      : Kulaklık        | kategori  : elektronik      | fiyat     :           250.0
 | urun      : Kitap           | kategori  : kirtasiye       | fiyat     :            45.0
 | urun      : Telefon Kilifi  | kategori  : elektronik      | fiyat     :            80.0
 | urun      : Defter          | kategori  : kirtasiye       | fiyat     :            15.0
 | urun      : Kulaklık        | kategori  : elektronik      | fiyat     :           250.0

KDV'li fiyatlar (map):

 | urun      : Kulaklık        | kategori  : elektronik      | fiyat     :           295.0
 | urun      : Kitap           | kategori  : kirtasiye       | fiyat     :            53.1
 | urun      : Telefon Kilifi  | kategori  : elektronik      | fiyat     :            94.4
 | urun      : Defter          | kategori  : kirtasiye       | fiyat     :            17.7
 | urun      : Kulaklık        | kategori  : elektronik      | fiyat     :           295.0

Sadece Elektronik (filter):

 | urun      : Ku

### 10 - Decoratorler

Bağlam: Bir ödeme API'sinde, dış servis çağrılarını saran fonksiyonlara çalışma süresi ölçümü ve otomatik yeniden deneme (retry) davranışı, fonksiyon kodunu değiştirmeden dekoratörlerle eklenmesi isteniyor. Gereksinimler:

1. sure_olc adında parametresiz bir dekoratör yaz; sardığı fonksiyonun çalışma süresini ölçüp "[SÜRE] fonksiyon_adı: X.XXXX sn" formatında yazdırsın, fonksiyonun orijinal dönüş değerini bozmadan geri döndürsün. functools.wraps kullanarak orijinal fonksiyonun __name__/__doc__ bilgisini koru.
2. yeniden_dene(max_deneme=3, beklenen_hata=Exception) adında parametreli bir dekoratör fabrikası yaz; sardığı fonksiyon beklenen_hata türünden bir istisna fırlatırsa, max_deneme sayısına kadar tekrar çağırsın; her başarısız denemede kaçıncı deneme olduğunu logla; tüm denemeler tükenirse son yakalanan istisnayı yeniden fırlatsın.
3. İki dekoratörü aynı fonksiyon üzerinde istifleyerek (stacking) kullan: @sure_olc en dışta, @yeniden_dene(max_deneme=3, beklenen_hata=ConnectionError) içeride olacak şekilde bir odeme_servisine_baglan(basarisiz_deneme_sayisi) fonksiyonuna uygula (bu fonksiyon test amaçlı, verilen sayı kadar önce hata fırlatıp sonra başarılı dönsün).
4. beklenen_hata dışında bir istisna türü fırlatılırsa (ValueError gibi) yeniden_dene bunu yakalamadan olduğu gibi yukarı fırlatmalı — yalnızca belirtilen hata türü retry mantığına girsin.
5. Dekore edilmiş fonksiyonun __name__ ve __doc__ özniteliklerinin functools.wraps sayesinde korunduğunu bir assert veya print ile doğrula.
6. max_deneme denemesinin hepsi tükenip son deneme de başarısız olursa, orijinal istisna tipinin ve mesajının korunarak dışarı fırlatıldığını göster.

In [ ]:
# Test Verileri
# Senaryo 1: 2 kez başarısız olup 3. denemede başarılı olan bağlantı
basarisiz_deneme_sayisi_1 = 2

# Senaryo 2: max_deneme'nin tükendiği, hep başarısız olan bağlantı
basarisiz_deneme_sayisi_2 = 10  # max_deneme=3 olduğundan hiçbir zaman başarıya ulaşmaz

# Senaryo 3: beklenmeyen hata türü (ValueError) - retry'a girmemeli
gecersiz_parametre = -1


# Çözüm
import time
import functools 

def sure_olc(fonksiyon):
    @functools.wraps(fonksiyon)
    def wrapper(*args, **kwargs):
        baslangic = time.perf_counter()
        sonuc = fonksiyon(*args, **kwargs)
        toplam_sure = time.perf_counter() - baslangic
        print(f"[SÜRE] {fonksiyon.__name__}: {toplam_sure:.4f} sn")
        return sonuc
    return wrapper 


def yeniden_dene(max_deneme=3, beklenen_hata=Exception):
    def decorator(fonksiyon):
        @functools.wraps(fonksiyon)
        def wrapper(*args, **kwargs):
            son_hata = None
            for deneme in range(1, max_deneme+1):
                try:
                    return fonksiyon(*args, **kwargs)
                except beklenen_hata as hata_mesaji:
                    son_hata = hata_mesaji
                    print(f"[RETRY] {fonksiyon.__name__} - deneme {deneme}/{max_deneme} başarısız: {hata_mesaji}")
            raise son_hata
        return wrapper
    return decorator

# Dekoratörleri kullanarak bir fonksiyon tanımlıyoruz
@sure_olc
@yeniden_dene(max_deneme=3, beklenen_hata=ConnectionError)
def odeme_servisine_baglan(basarisiz_deneme_sayisi):
    """Functools sayesinde metadata korunuyor. 
    Wrapper'ın datası esas func'ı override etmiyor."""

    # closure (global/nonlocal keywlerini) kullanmadan fonksiyona özellik bağlamak
    # ilk çalıştığında fonksiyonda "deneme_no" özelliğini bulamıyor ve kendi oluşturup bizim verdiğimiz 0
    # değerini o özelliğe atıyor. standart değer olan 0'ı atayıp sonra hemen 1 ekliyor burada
    odeme_servisine_baglan.deneme_no = getattr(odeme_servisine_baglan, "deneme_no", 0) + 1

    if odeme_servisine_baglan.deneme_no <= basarisiz_deneme_sayisi:
        raise ConnectionError(f"Bağlantı hatası (deneme {odeme_servisine_baglan.deneme_no})")

    if basarisiz_deneme_sayisi < 0:
        raise ValueError("basarisiz_deneme_sayisi negatif olamaz")

    return "Bağlantı başarılı"
      

print(f"Fonksiyon adı: {odeme_servisine_baglan.__name__}")
print(f"Fonksiyonun docstring'i: {odeme_servisine_baglan.__doc__}")
print("-" * 30)

# Test 1
odeme_servisine_baglan.deneme_no = 0    
sonuc = odeme_servisine_baglan(basarisiz_deneme_sayisi_1)
print(sonuc)

# Test 2
odeme_servisine_baglan.deneme_no = 0
try:
    odeme_servisine_baglan(basarisiz_deneme_sayisi_2)
except ConnectionError as hata_mesaji:
    print(f"Hata: {hata_mesaji}")

# Test 3
odeme_servisine_baglan.deneme_no = 0
try:
    odeme_servisine_baglan(gecersiz_parametre)
except ValueError as hata_mesaji:
    print(f"Hata: {hata_mesaji}") 

            

Fonksiyon adı: odeme_servisine_baglan
Fonksiyonun docstring'i: Functools sayesinde metadata korunuyor. 
Wrapper'ın datası esas func'ı override etmiyor.
------------------------------
[RETRY] odeme_servisine_baglan - deneme 1/3 başarısız: Bağlantı hatası (deneme 1)
[RETRY] odeme_servisine_baglan - deneme 2/3 başarısız: Bağlantı hatası (deneme 2)
[SÜRE] odeme_servisine_baglan: 0.0001 sn
Bağlantı başarılı
[RETRY] odeme_servisine_baglan - deneme 1/3 başarısız: Bağlantı hatası (deneme 1)
[RETRY] odeme_servisine_baglan - deneme 2/3 başarısız: Bağlantı hatası (deneme 2)
[RETRY] odeme_servisine_baglan - deneme 3/3 başarısız: Bağlantı hatası (deneme 3)
Hata: Bağlantı hatası (deneme 3)
Hata: basarisiz_deneme_sayisi negatif olamaz
